# Project: BBLF AI Selector v2
# Section: Optimisation & Simulation Tool
# Sub Section: Pre Tournament

Goal: Select the optimal starting 12 players prior to the start of the tournament 

Things to add:
1. Additional constraints to capture fantasy nuances e.g. captain points, vice captain trick
2. Advance simulation based optimisation process 
2. Nice to have: bench tricks

# 0. Prerequistes

In [35]:
pip install mip==1.16rc0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
# 0. Prerequistes

import pandas as pd
import numpy as np
import os
import random
from mip import Model, xsum, maximize, BINARY 
from scipy.stats import norm

os.getcwd()
directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2'
add_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/pre_tourny/'
over_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/overall'
py_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/python_data/'
mock_data_dir = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/mock_data/'

In [37]:
# Optimisation Strategy
opt_strat = 'efp' # Options: 'efp' (player expected fantasy points), 'range' (player fantasy point range)

# 1. Data Extraction 

In [38]:
if opt_strat == 'range':
    # Simulated Player Points Data Constructions
    # Extract Player Expected Points and Standard Deviation dataframe
    model_pts_df = pd.read_csv(os.path.join(mock_data_dir,'simulated_mock_data.csv'), low_memory=False)
    sim_pts_df = model_pts_df.copy()

    # Generate Simulated Points based on Normal Distribution
    # Function to calculate z score bounds given confidence interval
    def z_score_bounds(confidence_level):
        """
        Returns the lower and upper z-scores for a given confidence level.
        For example, confidence_level = 0.90 for 90%.
        """
        alpha = 1 - confidence_level
        lower = norm.ppf(alpha / 2)
        upper = norm.ppf(1 - alpha / 2)
        return lower, upper

    # Set confidence interval and calculate z score bounds
    conf_int = 0.9
    lower_z_thresh, upper_z_thresh = z_score_bounds(conf_int)
    print(f"For {conf_int*100:.0f}% simulated points confidence interval the lower z score is {lower_z_thresh:.3f} and upper z score is {upper_z_thresh:.3f}")

    # Assign random z score for each df row within bounds and calculate simulated points
    sim_pts_df["z_score"] = np.random.uniform(lower_z_thresh, upper_z_thresh, size=len(sim_pts_df))
    sim_pts_df["sim_points"] = sim_pts_df["mean"] + (sim_pts_df["z_score"] * sim_pts_df["std_dev"])
    sim_pts_df["sim_points"] = sim_pts_df["sim_points"].clip(lower=0).round(0)  # Ensure no negative points

else:
    print("Change opt_strat to range for player fp distribution Optimisation Strategy")

Change opt_strat to range for player fp distribution Optimisation Strategy


In [39]:
# New Optimisation Strategy (Player Fantasy Point Distribution)
if opt_strat == 'range':
    # Data Extraction 
    # Current Round & Optimal Number of Rounds
    current_round = 1
    opt_round = 2

    # Pull in player price data csv file 
    price_df = pd.read_csv(os.path.join(add_data_directory,'player_price_2026.csv'), low_memory=False)

    # Player Role Flags
    price_df["Wk_f"] = np.where((price_df["Role"] == "WK"), 1, 0)
    price_df["Bat_f"] = np.where((price_df["Role"] == "WK") |(price_df["Role"] == "BAT") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df["Bowl_f"] = np.where((price_df["Role"] == "BOWL") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df = price_df[["Full_Name", "Price", "Team","Wk_f", "Bat_f", "Bowl_f", "Role", "Available", "In_Team"]].rename(columns = {"Full_Name": "Name"}) 

    team_fix_df = pd.read_csv(os.path.join(over_data_directory,'team_loc_fixture.csv'), low_memory=False)
    # team_fix_df = team_fix_df[team_fix_df.Round >= current_round].dropna()
    # team_fix_df = team_fix_df[team_fix_df.Round <= current_round + opt_round - 1].dropna()

    # Join team fixture to player price to create a row for each game
    game_df = pd.merge(price_df, team_fix_df, left_on = ["Team"], right_on = ["Team"], how = "left")

    # Pull player expected points csv file
    # sim_pts_df = pd.read_csv(os.path.join(mock_data_dir,'simulated_mock_data.csv'), low_memory=False).drop(["Unnamed: 0", "player"], axis = 1)
    sim_pts_df = pd.read_csv(os.path.join(mock_data_dir,'simulated_mock_data.csv'), low_memory=False)

    # Join on expected points for each game
    player_df_raw = pd.merge(game_df, sim_pts_df, left_on = ["Name", "Team", "Opposition", "Venue", "Home_f","Round"], right_on = ["Name", "Team", "Opposition", "Venue", "Home_f","Round"], how = "left")
    player_df_raw["sim_points"] = np.where((player_df_raw["Role"] == "WK"), player_df_raw["sim_points"] + 10.51, player_df_raw["sim_points"] + 4.53)
    player_df_raw["weight"] = 1

    # Aggregate by player name (calculate total expected points for each player for x number of rounds)
    player_df = player_df_raw.groupby(['Name', 'Price', "Team", "Wk_f", "Bat_f", "Bowl_f", "Role", "weight","Available", "In_Team"], as_index=False).agg(
    sim_points=('sim_points',"sum"))

    # Create dataframe for next round expected points (For captain selection)
    player_df_next_rnd = player_df_raw[player_df_raw.Round == current_round].groupby(['Name', 'Price', "Team", "Wk_f", "Bat_f", "Bowl_f", "Role", "weight","Available", "In_Team"], as_index=False).agg(
    next_rnd_sim_points=('sim_points',"sum"))
    player_df_next_rnd = player_df_next_rnd[["Name", "Team", "next_rnd_sim_points"]]

else:
    print("Change opt_strat to range for player fp distribution Optimisation Strategy")


Change opt_strat to range for player fp distribution Optimisation Strategy


In [40]:
# Previous Season Optimisation Strategy (Expected Fantasy Points)
if opt_strat == 'efp':
    
    # Data Extraction 
    # Current Round & Optimal Number of Rounds
    current_round = 1
    opt_round = 3

    # Pull in player price data csv file 
    price_df = pd.read_csv(os.path.join(add_data_directory,'player_price_2026.csv'), low_memory=False)

    # Create Player Role Flags
    price_df["Wk_f"] = np.where((price_df["Role"] == "WK"), 1, 0)
    price_df["Bat_f"] = np.where((price_df["Role"] == "WK") |(price_df["Role"] == "BAT") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df["Bowl_f"] = np.where((price_df["Role"] == "BOWL") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df = price_df[["Full_Name", "Price", "Team","Wk_f", "Bat_f", "Bowl_f", "Role", "Available", "In_Team"]].rename(columns = {"Full_Name": "Name"}) 

    team_fix_df = pd.read_csv(os.path.join(over_data_directory,'team_loc_fixture.csv'), low_memory=False)
    team_fix_df = team_fix_df[team_fix_df.Round >= current_round].dropna()
    team_fix_df = team_fix_df[team_fix_df.Round <= current_round + opt_round - 1].dropna()

    # Join team fixture to player price to create a row for each game
    game_df = pd.merge(price_df , team_fix_df, left_on = ["Team"], right_on = ["Team"], how = "left")

    # Pull player expected points csv file
    efp_df = pd.read_csv(os.path.join(py_data_directory,'bbl15_efp_model_score_pre_tourny_short.csv'), low_memory=False)

    # Join on expected points for each game
    player_df_raw = pd.merge(game_df, efp_df, left_on = ["Name", "Team", "Opposition", "Venue", "Home_f","Round"], right_on = ["Name", "Team", "opp", "venue", "Home_f","Round"], how = "left")
    player_df_raw["exp_points"] = np.where((player_df_raw["Role"] == "WK"), player_df_raw["exp_pts"] + 10.51, player_df_raw["exp_pts"] + 4.53)
    player_df_raw["weight"] = 1

    # Aggregate by player name (calculate total expected points for each player for x number of rounds)
    player_df = player_df_raw.groupby(['Name', 'Price', "Team", "Wk_f", "Bat_f", "Bowl_f", "Role", "weight","Available", "In_Team"], as_index=False).agg(
    exp_points=('exp_points',"sum"))

    # Create dataframe for next round expected points (For captain selection)
    player_df_next_rnd = player_df_raw[player_df_raw.Round == current_round].groupby(['Name', 'Price', "Team", "Wk_f", "Bat_f", "Bowl_f", "Role", "weight","Available", "In_Team"], as_index=False).agg(
    next_rnd_exp_points=('exp_points',"sum"))
    player_df_next_rnd = player_df_next_rnd[["Name", "Team", "next_rnd_exp_points"]]

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

# 2. Optimisation Process (Pre Tournament Selection) - Starting 12 Players

Optimisation Objective: Maximise the number of expected fantasy points

Constraints: 
1. Number of players selected must be 12
2. Atleast 1 wicketkeeper
3. Atleast 6 batters
4. Atleast 5 bowlers
5. Total budget of team is less than $1,783,500 (As bench costs $216,500)

In [41]:
# EFP Optimisation Variable Setup
if opt_strat == 'efp':
    points = player_df["exp_points"]
    price = player_df["Price"]

    weight = player_df["weight"]
    available = player_df["Available"]
    in_team = player_df["In_Team"]
    wk_weight = player_df["Wk_f"]
    bat_weight = player_df["Bat_f"]
    bowl_weight = player_df["Bowl_f"]

    play_cnt, total_player = 12, range(len(price))
    # team_play_cnt, total_team_player = 0, range(len(price)) # IGNORE as pre tourny no players in team
    wk_cnt, total_wk = 1, range(len(price))
    bat_cnt, total_bat = 6, range(len(price))
    bowl_cnt, total_bowl = 5, range(len(price))
    budget, total_budget = 1783500, range(len(price))

    # MIP Optimsation Setup
    m = Model("knapsack")
    x = [m.add_var(var_type=BINARY) for i in total_player]
    print(x)

    m.objective = maximize(xsum(points[i]*x[i] for i in total_player))

    m += xsum(weight[i] * x[i] for i in total_player) == play_cnt
    m += xsum(wk_weight[i] * x[i] for i in total_wk) >= wk_cnt
    m += xsum(bat_weight[i] * x[i] for i in total_bat) >= bat_cnt
    m += xsum(bowl_weight[i] * x[i] for i in total_bowl) >= bowl_cnt
    m += xsum(price[i] * x[i] for i in total_budget) <= budget
    m += xsum(available[i] * x[i] for i in total_player) == play_cnt
    # m += xsum(in_team[i] * x[i] for i in total_team_player) >= team_play_cnt

    # Optimisation Process & Results
    m.optimize() 
    selected = [i for i in total_player if x[i].x >= 0.99]
    print("selected items: {}".format(selected))

    # Optimal Team Output
    sel_player_df = player_df.iloc[selected]
    sel_player_df = pd.merge(sel_player_df, player_df_next_rnd, left_on = ["Name","Team"], right_on = ["Name","Team"], how = "left")
    print("Total Expected Points:", sum(sel_player_df["exp_points"]))
    print("Total Next Round Points:", sum(sel_player_df["next_rnd_exp_points"]))
    print("Total Team Cost:", sum(sel_player_df["Price"]))
    print("Number of Wk:", sum(sel_player_df["Wk_f"]))
    print("Number of Bat:", sum(sel_player_df["Bat_f"]))
    print("Number of Bowl:", sum(sel_player_df["Bowl_f"]))
    print("Available Players:", sum(sel_player_df["Available"]))
    print("Current Players Remaining:", sum(sel_player_df["In_Team"]))

    print(sel_player_df.sort_values(by = "Price", ascending = False))
    print(sel_player_df[sel_player_df['In_Team'] == 1].sort_values(by = "Price", ascending = False))
    print(sel_player_df[sel_player_df['In_Team'] == 0].sort_values(by = "Price", ascending = False))

    # Save Optimal Team to CSV
    sel_player_df.to_csv(os.path.join(add_data_directory,'optim/EFP_optimal_pre_tourny_team.csv'))

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

[<mip.entities.Var object at 0x000002934F205D50>, <mip.entities.Var object at 0x000002934F207FD0>, <mip.entities.Var object at 0x000002934F205A80>, <mip.entities.Var object at 0x000002934F204850>, <mip.entities.Var object at 0x000002934F205690>, <mip.entities.Var object at 0x000002934F204100>, <mip.entities.Var object at 0x000002934F204C40>, <mip.entities.Var object at 0x000002934F2057E0>, <mip.entities.Var object at 0x000002934F2055A0>, <mip.entities.Var object at 0x000002934F204E80>, <mip.entities.Var object at 0x000002934F2055D0>, <mip.entities.Var object at 0x000002934F205750>, <mip.entities.Var object at 0x000002934F204580>, <mip.entities.Var object at 0x000002934F207970>, <mip.entities.Var object at 0x000002934F205480>, <mip.entities.Var object at 0x000002934F205660>, <mip.entities.Var object at 0x000002934F205F30>, <mip.entities.Var object at 0x000002934F2049A0>, <mip.entities.Var object at 0x000002934F207DF0>, <mip.entities.Var object at 0x000002934F207730>, <mip.entities.Var o

In [42]:
# Player Distribution Optimisation Variable Setup
if opt_strat == 'range':
    points = player_df["sim_points"]
    price = player_df["Price"]

    weight = player_df["weight"]
    available = player_df["Available"]
    in_team = player_df["In_Team"]
    wk_weight = player_df["Wk_f"]
    bat_weight = player_df["Bat_f"]
    bowl_weight = player_df["Bowl_f"]

    play_cnt, total_player = 12, range(len(price))
    # team_play_cnt, total_team_player = 0, range(len(price)) # IGNORE as pre tourny no players in team
    wk_cnt, total_wk = 1, range(len(price))
    bat_cnt, total_bat = 6, range(len(price))
    bowl_cnt, total_bowl = 5, range(len(price))
    budget, total_budget = 1783500, range(len(price))

    # MIP Optimsation Setup
    m = Model("knapsack")
    x = [m.add_var(var_type=BINARY) for i in total_player]
    print(x)

    m.objective = maximize(xsum(points[i]*x[i] for i in total_player))

    m += xsum(weight[i] * x[i] for i in total_player) == play_cnt
    m += xsum(wk_weight[i] * x[i] for i in total_wk) >= wk_cnt
    m += xsum(bat_weight[i] * x[i] for i in total_bat) >= bat_cnt
    m += xsum(bowl_weight[i] * x[i] for i in total_bowl) >= bowl_cnt
    m += xsum(price[i] * x[i] for i in total_budget) <= budget
    m += xsum(available[i] * x[i] for i in total_player) == play_cnt
    # m += xsum(in_team[i] * x[i] for i in total_team_player) >= team_play_cnt

    # Optimisation Process & Results
    m.optimize() 
    selected = [i for i in total_player if x[i].x >= 0.99]
    print("selected items: {}".format(selected))

    # Optimal Team Output
    sel_player_df = player_df.iloc[selected]
    sel_player_df = pd.merge(sel_player_df, player_df_next_rnd, left_on = ["Name","Team"], right_on = ["Name","Team"], how = "left")
    print("Total Simulated Points:", sum(sel_player_df["sim_points"]))
    print("Total Next Round Points:", sum(sel_player_df["next_rnd_sim_points"]))
    print("Total Team Cost:", sum(sel_player_df["Price"]))
    print("Number of Wk:", sum(sel_player_df["Wk_f"]))
    print("Number of Bat:", sum(sel_player_df["Bat_f"]))
    print("Number of Bowl:", sum(sel_player_df["Bowl_f"]))
    print("Available Players:", sum(sel_player_df["Available"]))
    print("Current Players Remaining:", sum(sel_player_df["In_Team"]))

    print(sel_player_df.sort_values(by = "Price", ascending = False))
    print(sel_player_df[sel_player_df['In_Team'] == 1].sort_values(by = "Price", ascending = False))
    print(sel_player_df[sel_player_df['In_Team'] == 0].sort_values(by = "Price", ascending = False))

    # Save Optimal Team to CSV
    sel_player_df.to_csv(os.path.join(add_data_directory,'EFP_optimal_pre_tourny_team.csv'))

else:
    print("Change opt_strat to range for player distribution Optimisation Strategy")

Change opt_strat to range for player distribution Optimisation Strategy
